# Фінальний проєкт. Реляційні бази даних

**Структура:**
- Підготовка: запуск PostgreSQL і допоміжна функція
- Завдання 1. Завантаження даних (схема `pandemic`, імпорт CSV, огляд)
- Завдання 2. Нормалізація до 3НФ
- Завдання 3. Агрегати для `number_rabies`
- Завдання 4. Колонка різниці в роках (вбудовані функції)
- Завдання 5. Власні SQL-функції

## Підготовка. Запуск PostgreSQL

- `pgserver` запускає PostgreSQL без окремої інсталяції. Каталог бази — **новий тимчасовий каталог** (`tempfile.mkdtemp`), бо macOS періодично чистить `/tmp` і база у фіксованому каталозі ламається. `cleanup_mode='delete'` видаляє базу після зупинки kernel, тому весь результат відтворюється кодом цього notebook.
- Драйвер `postgresql+psycopg2://` вказано явно: SQLAlchemy 2.1 за замовчуванням шукає psycopg 3, а для імпорту потрібен `copy_expert` із psycopg2.

In [1]:
import tempfile

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

PG_DIR = tempfile.mkdtemp(prefix='fp_pg_')
pg = pgserver.get_server(PG_DIR, cleanup_mode='delete')

uri = pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(uri)

pd.read_sql(text('SELECT version()'), engine)

,version
0,"PostgreSQL 16.2 on aarch64-apple-darwin23.5.0,..."


Допоміжна функція `run_sql` виконує SQL (можна кілька команд підряд) у транзакції: `engine.begin()` робить COMMIT наприкінці блоку або ROLLBACK у разі помилки. Якщо запит повертає рядки, результат повертається як DataFrame.

In [2]:
def run_sql(sql: str):
    """Виконує SQL (можна кілька команд) з COMMIT; повертає DataFrame, якщо є результат."""
    with engine.begin() as conn:
        result = conn.execute(text(sql))
        if result.returns_rows:
            return pd.DataFrame(result.fetchall(), columns=list(result.keys()))

## Завдання 1. Завантаження даних

### 1.1. Схема `pandemic` і схема за замовчуванням

- **Схема** — іменований простір («папка») для таблиць усередині бази; без неї все потрапляє в `public`.
- `CREATE SCHEMA IF NOT EXISTS` — повторний запуск не дає помилки.
- `SET search_path TO pandemic` робить `pandemic` схемою за замовчуванням: імена таблиць без префікса шукаються в ній (аналог `USE` у MySQL). `SET` діє в межах сесії (підключення); SQLAlchemy повторно використовує те саме підключення з пулу, а в `COPY` нижче ім'я таблиці для надійності вказано повністю.
- `current_schema()` підтверджує результат.

In [3]:
run_sql("""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;

SELECT current_schema();
""")

,current_schema
0,pandemic


### 1.2. Таблиця `infectious_cases`

Структура повторює CSV (12 колонок у тому самому порядку — `COPY` зіставляє колонки за позицією). Назви — малими літерами без лапок, щоб у запитах не писати `"Number_rabies"`.

| Колонка | Тип | Чому |
|---|---|---|
| `entity` | `TEXT NOT NULL` | назва країни/регіону, порожніх немає |
| `code` | `TEXT` | ISO-код країни; у регіонів (`Africa`, `G20`, …) коду немає → NULL |
| `year` | `INTEGER NOT NULL` | рік — ціле число (знадобиться для `make_date` у завданнях 4–5) |
| показники хвороб | `DOUBLE PRECISION` | `number_rabies`, `number_malaria`, `number_hiv`, `number_tuberculosis` — дробові модельні оцінки (напр. `2.145772`), з `INTEGER` імпорт упав би |

`DROP TABLE IF EXISTS` робить клітинку перезапускною.

In [4]:
run_sql("""
DROP TABLE IF EXISTS infectious_cases;

CREATE TABLE infectious_cases (
    entity               TEXT    NOT NULL,
    code                 TEXT,
    year                 INTEGER NOT NULL,
    number_yaws          DOUBLE PRECISION,
    polio_cases          DOUBLE PRECISION,
    cases_guinea_worm    DOUBLE PRECISION,
    number_rabies        DOUBLE PRECISION,
    number_malaria       DOUBLE PRECISION,
    number_hiv           DOUBLE PRECISION,
    number_tuberculosis  DOUBLE PRECISION,
    number_smallpox      DOUBLE PRECISION,
    number_cholera_cases DOUBLE PRECISION
);
""")

### 1.3. Імпорт CSV через `COPY`

- `engine.raw_connection()` — сиря psycopg2-підключення, бо `copy_expert` є лише в psycopg2;
- `FORMAT csv, HEADER true` — пропустити рядок заголовків; порожнє поле (`,,`) у CSV-режимі стає **NULL**;
- `raw_conn.commit()` — фіксує завантаження; `try/finally` закриває підключення навіть у разі помилки.

In [5]:
raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open('infectious_cases.csv', 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                """
                COPY pandemic.infectious_cases
                FROM STDIN
                WITH (FORMAT csv, HEADER true, DELIMITER ',')
                """,
                file,
            )
    raw_conn.commit()
finally:
    raw_conn.close()

run_sql('SELECT COUNT(*) AS loaded_rows FROM infectious_cases')

,loaded_rows
0,10521


### 1.4. Огляд даних

In [6]:
run_sql("""
SELECT *
FROM infectious_cases
LIMIT 10;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160.0,0.0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859.0,0.0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730.0,0.0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937.0,0.0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864.0,0.0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867.0,0.0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901.0,0.0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396.0,0.0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149.0,0.0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385.0,0.0,None,None,None,None,None,None


**Обсяг даних і пропуски.** `COUNT(*)` рахує всі рядки, `COUNT(колонка)` — лише рядки, де значення не NULL; різниця дає кількість пропусків.

In [7]:
run_sql("""
SELECT
    COUNT(*)              AS total_rows,
    COUNT(number_rabies)  AS rabies_not_null,
    COUNT(*) - COUNT(number_rabies) AS rabies_null,
    COUNT(code)           AS code_not_null
FROM infectious_cases;
""")

,total_rows,rabies_not_null,rabies_null,code_not_null
0,10521,6780,3741,9393


**Унікальні значення і період.** `COUNT(DISTINCT (entity, code))` рахує унікальні **пари**; `MIN`/`MAX(year)` — період, який покривають дані.

In [8]:
run_sql("""
SELECT
    COUNT(DISTINCT entity)          AS unique_entities,
    COUNT(DISTINCT (entity, code))  AS unique_entity_code,
    MIN(year)                       AS first_year,
    MAX(year)                       AS last_year
FROM infectious_cases;
""")

,unique_entities,unique_entity_code,first_year,last_year
0,245,245,1920,2023


**Рядки без коду країни.** NULL перевіряється через `IS NULL` (порівняння `= NULL` завжди дає NULL, а не TRUE). Без коду — 34 записи: переважно агрегати, а також частини країн на кшталт `England`.

In [9]:
run_sql("""
SELECT entity, COUNT(*) AS n_rows
FROM infectious_cases
WHERE code IS NULL
GROUP BY entity
ORDER BY n_rows DESC;
""")

,entity,n_rows
0,Asia,73
1,Africa,52
2,Europe,51
3,Americas,41
4,Oceania,41
5,Western Pacific,41
6,South-East Asia,41
7,Eastern Mediterranean,41
8,North America,34
9,Latin America & Caribbean (WB),30


### Висновки до завдання 1

- Схему `pandemic` створено й обрано за замовчуванням; таблицю `infectious_cases` завантажено повністю — **10 521** рядок, 12 колонок.
- Дані покривають **1920–2023** роки і **245** країн/регіонів; `number_rabies` заповнено в **6 780** рядках, у **3 741** — NULL.
- Унікальних `entity` стільки ж, скільки пар `(entity, code)` (245): кожна назва має рівно один код (або жодного), тобто `code` функціонально залежить від `entity`. Через це `entity` і `code` дублюються в кожному рядку.

## Завдання 2. Нормалізація до 3-ї нормальної форми

**Проблема.** У `infectious_cases` пара `entity, code` повторюється в кожному рядку (для кожного року): `Afghanistan, AFG` записано десятки разів. Це надлишковість і ризик **аномалії оновлення** — змінивши код в одному рядку й забувши інші, отримаємо суперечливі дані.

**Рішення** — винести пару `entity, code` в окрему таблицю `entities`, а в таблиці показників `disease_cases` зберігати лише посилання `entity_id`:

```
entities (id PK, entity, code)  ◀──  disease_cases (id PK, entity_id FK, year, number_*)
```

### 2.1. Таблиця унікальних комбінацій `entity` та `code`

- `DROP` у порядку «дочірня → батьківська»: `disease_cases` посилається на `entities`, тому видаляється першою.
- `SERIAL PRIMARY KEY` — автоматичний лічильник `id`; первинний ключ унікальний, не NULL, і на нього можна посилатися.
- `UNIQUE NULLS NOT DISTINCT (entity, code)` — пара не може повторитися. Без `NULLS NOT DISTINCT` (PostgreSQL 15+) два NULL вважаються різними, і `('Africa', NULL)` можна було б вставити двічі.
- `INSERT ... SELECT DISTINCT` заповнює таблицю результатом запиту; `ORDER BY entity` робить `id` детермінованими (за алфавітом) при кожному запуску.

In [10]:
run_sql("""
DROP TABLE IF EXISTS disease_cases;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (
    id     SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code   TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, code
FROM infectious_cases
ORDER BY entity;

SELECT * FROM entities ORDER BY id LIMIT 10;
""")

,id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,NaN
2,3,African Region (WHO),NaN
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,NaN
7,8,Andorra,AND
8,9,Angola,AGO
9,10,Antigua and Barbuda,ATG


### 2.2. Таблиця показників з `entity_id` замість `entity` і `code`

- `entity_id ... REFERENCES entities(id)` — **зовнішній ключ**: база не дозволить посилання на неіснуючу країну, зв'язок гарантований на рівні СУБД.
- `UNIQUE (entity_id, year)` — один рядок на країну за рік (так і є у вихідних даних).
- `JOIN` підставляє для кожного вихідного рядка відповідний `id` з `entities`.
- **`IS NOT DISTINCT FROM` замість `=` для `code`**: у регіонів (`Africa`, `G20`, …) `code` дорівнює NULL, а `NULL = NULL` дає NULL, не TRUE, тож звичайний `JOIN` мовчки відкинув би ці рядки — у таблицю потрапило б лише 9 393 рядки з 10 521. `IS NOT DISTINCT FROM` вважає два NULL рівними.

In [11]:
run_sql("""
CREATE TABLE disease_cases (
    id                   SERIAL PRIMARY KEY,
    entity_id            INTEGER NOT NULL REFERENCES entities(id),
    year                 INTEGER NOT NULL,
    number_yaws          DOUBLE PRECISION,
    polio_cases          DOUBLE PRECISION,
    cases_guinea_worm    DOUBLE PRECISION,
    number_rabies        DOUBLE PRECISION,
    number_malaria       DOUBLE PRECISION,
    number_hiv           DOUBLE PRECISION,
    number_tuberculosis  DOUBLE PRECISION,
    number_smallpox      DOUBLE PRECISION,
    number_cholera_cases DOUBLE PRECISION,
    UNIQUE (entity_id, year)
);

INSERT INTO disease_cases (
    entity_id, year,
    number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
SELECT
    e.id, ic.year,
    ic.number_yaws, ic.polio_cases, ic.cases_guinea_worm, ic.number_rabies,
    ic.number_malaria, ic.number_hiv, ic.number_tuberculosis,
    ic.number_smallpox, ic.number_cholera_cases
FROM infectious_cases AS ic
JOIN entities AS e
  ON  e.entity = ic.entity
  AND e.code IS NOT DISTINCT FROM ic.code;

SELECT COUNT(*) AS n_rows FROM disease_cases;
""")

,n_rows
0,10521


### 2.3. Перевірка: нормалізація без втрат

У `entities` має бути стільки рядків, скільки унікальних пар `(entity, code)` в оригіналі (245), а в `disease_cases` — стільки ж рядків і заповнених значень `number_rabies`, скільки в `infectious_cases` (10 521 і 6 780).

In [12]:
run_sql("""
SELECT
    COUNT(*) AS total_rows
FROM entities;
""")

,total_rows
0,245


In [13]:
run_sql("""
SELECT
    COUNT(*)              AS total_rows,
    COUNT(number_rabies)  AS rabies_not_null
FROM disease_cases;
""")

,total_rows,rabies_not_null
0,10521,6780


### Висновки до завдання 2

- У схемі `pandemic` створено дві нормалізовані таблиці: **`entities`** — 245 унікальних комбінацій `entity, code`, та **`disease_cases`** — 10 521 рядок показників, де замість повторюваних `entity` і `code` зберігається `entity_id`.
- Зв'язок забезпечено зовнішнім ключем `disease_cases.entity_id → entities.id`; обмеження `UNIQUE` фіксують правила даних (унікальна пара `entity, code`; один рядок на країну за рік).
- Кількість рядків і заповнених значень `number_rabies` збігається з оригіналом — дані перенесено без втрат, зокрема 1 128 рядків регіонів без коду (завдяки `IS NOT DISTINCT FROM`).
- Таблиці перебувають у 3НФ: `code` тепер залежить лише від ключа `entities.id`, а показники в `disease_cases` — від ключа `(entity_id, year)`.

## Завдання 3. Аналіз `number_rabies`

Для кожної унікальної комбінації `entity, code` (тобто для кожного `entities.id`) рахуємо середнє, мінімум, максимум і суму `number_rabies`, сортуємо за середнім за спаданням і виводимо 10 рядків.

- **`GROUP BY e.id, e.entity, e.code`** — одна група на країну/регіон; `entity` і `code` перелічено в `GROUP BY`, бо вони є в `SELECT` поза агрегатними функціями.
- **`AVG`, `MIN`, `MAX`, `SUM`** згортають кожну групу в одне число; NULL вони ігнорують. `COUNT(number_rabies)` додатково показує, зі скількох років пораховано показники.
- **`JOIN entities`** — у нормалізованій таблиці `disease_cases` лише `entity_id`, назву й код беремо з `entities`.
- **`WHERE dc.number_rabies IS NOT NULL`** — враховуємо лише числові значення. Без фільтра 19 країн/регіонів, у яких `number_rabies` порожній в усіх роках, отримали б `AVG = NULL`, а PostgreSQL під час `ORDER BY ... DESC` ставить NULL **першими** — топ-10 складався б із порожніх значень. `WHERE` відкидає такі рядки ще до групування.
- Порядок виконання: `FROM/JOIN → WHERE → GROUP BY → агрегати → ORDER BY → LIMIT`.

In [14]:
run_sql("""
SELECT
    e.id,
    e.entity,
    e.code,
    AVG(dc.number_rabies) AS avg_rabies,
    MIN(dc.number_rabies) AS min_rabies,
    MAX(dc.number_rabies) AS max_rabies,
    SUM(dc.number_rabies) AS sum_rabies,
    COUNT(dc.number_rabies) AS count_years
FROM disease_cases AS dc
JOIN entities AS e ON e.id = dc.entity_id
WHERE dc.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10;
""")

,id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies,count_years
0,241,World,OWID_WRL,20192.370367,14075.5080,24744.6580,605771.1110,30
1,121,Lower Middle Income (WB),NaN,15193.959383,10202.5280,19182.7950,455818.7815,30
2,202,South Asia (WB),NaN,11729.888753,7271.2754,15361.8780,351896.6626,30
3,206,South-East Asia Region (WHO),NaN,11424.326893,6806.0127,15641.9580,342729.8068,30
4,77,G20,NaN,10189.046003,6339.0796,13164.8810,305671.3801,30
5,97,India,IND,8599.173280,5425.8726,11121.1420,257975.1984,30
6,209,Sub-Saharan Africa (WB),NaN,4748.268090,4356.8184,5017.9770,142448.0427,30
7,3,African Region (WHO),NaN,4649.702377,4283.0730,4917.7840,139491.0713,30
8,120,Low Income (WB),NaN,3568.105450,3014.6170,3873.5330,107043.1635,30
9,58,East Asia & Pacific (WB),NaN,3427.461133,2009.9395,4591.0107,102823.8340,30


### Висновки до завдання 3

- Найбільше середнє значення `number_rabies` — у `World` (≈ 20 192 на рік); далі йдуть агрегати за рівнем доходу та регіонами (`Lower Middle Income (WB)`, `South Asia (WB)`, `South-East Asia Region (WHO)`, `G20`).
- Єдина окрема країна в топ-10 — **India** (6-те місце, середнє ≈ 8 599): регіональні суми закономірно більші за будь-яку країну всередині них. Умова завдання не вимагає виключати агрегати, тож результат наведено для всіх комбінацій `entity, code`.
- Для всіх лідерів показники пораховано за 30 років: дані про сказ у датасеті є лише за 1990–2019 роки.

## Завдання 4. Колонка різниці в роках

Для колонки `year` нормалізованої таблиці `disease_cases` будуємо вбудованими функціями PostgreSQL три атрибути:

- `AGE(пізніша, рання)` повертає різницю як календарний **інтервал** (`66 years 10 mons 6 days`), тобто враховує різну довжину місяців і високосні роки — на відміну від `CURRENT_DATE - дата`, що дає лише кількість днів. Порядок аргументів важливий: спершу пізніша дата.
- `EXTRACT(YEAR FROM ...)` бере з інтервалу лише кількість повних років; `EXTRACT` повертає `NUMERIC`, тому `::INTEGER` приводить результат до цілого числа.
- `age_interval` додано для наочності: pandas показує `INTERVAL` як кількість днів (`24396 days`), бо psycopg2 перетворює його на Python `timedelta`, який зберігає лише дні й секунди; у самій базі це `66 years 10 mons 6 days`.
- `ORDER BY e.entity, dc.year` робить вивід детермінованим.

In [15]:
run_sql("""
SELECT
    e.entity,
    dc.year,
    MAKE_DATE(dc.year, 1, 1) AS year_start_date,
    CURRENT_DATE             AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(dc.year, 1, 1)))::INTEGER AS years_diff,
    AGE(CURRENT_DATE, MAKE_DATE(dc.year, 1, 1)) AS age_interval
FROM disease_cases AS dc
JOIN entities AS e ON e.id = dc.entity_id
ORDER BY e.entity, dc.year
LIMIT 10;
""")

,entity,year,year_start_date,today,years_diff,age_interval
0,Afghanistan,1960,1960-01-01,2026-11-07,66,24396 days
1,Afghanistan,1965,1965-01-01,2026-11-07,61,22571 days
2,Afghanistan,1980,1980-01-01,2026-11-07,46,17096 days
3,Afghanistan,1981,1981-01-01,2026-11-07,45,16731 days
4,Afghanistan,1982,1982-01-01,2026-11-07,44,16366 days
5,Afghanistan,1983,1983-01-01,2026-11-07,43,16001 days
6,Afghanistan,1984,1984-01-01,2026-11-07,42,15636 days
7,Afghanistan,1985,1985-01-01,2026-11-07,41,15271 days
8,Afghanistan,1986,1986-01-01,2026-11-07,40,14906 days
9,Afghanistan,1987,1987-01-01,2026-11-07,39,14541 days


### Висновки до завдання 4

- Дату 1 січня, поточну дату і різницю в роках побудовано лише вбудованими функціями (`MAKE_DATE`, `CURRENT_DATE`, `AGE`, `EXTRACT`).
- Результат залежить від дня запуску: наведений output пораховано на **2026-11-07** (наприклад, 1960 → 66 років). У всій таблиці `years_diff` — від 3 (2023 рік) до 106 (1920 рік).
- Оскільки відлік іде від 1 січня, `years_diff` збігається з `рік_сьогодні − year`; `AGE` дає календарно точну різницю і для будь-якої іншої дати в році.

## Завдання 5. Власні SQL-функції

Дві незалежні функції в схемі `pandemic` (створюються в першій схемі з `search_path`):

### 5.1. `fn_years_since(year)` — різниця в роках між поточною датою та 1 січня року

Тіло — той самий вираз, що в завданні 4: `MAKE_DATE` → `AGE` → `EXTRACT(YEAR ...)::INTEGER`. Функція позначена `STABLE`, а не `IMMUTABLE`, бо залежить від `CURRENT_DATE`. Приклад з умови: 1996 → 1996-01-01 → 30 років (на 2026-11-07).

In [16]:
run_sql("""
CREATE OR REPLACE FUNCTION fn_years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;

SELECT fn_years_since(1996) AS example_1996;
""")

,example_1996
0,30


**Використання у запиті та перевірка:** результат функції порівнюється з колонкою, побудованою вбудованими функціями в завданні 4 — значення збігаються.

In [17]:
run_sql("""
SELECT
    e.entity,
    dc.year,
    fn_years_since(dc.year) AS years_diff_fn,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(dc.year, 1, 1)))::INTEGER AS years_diff_builtin
FROM disease_cases AS dc
JOIN entities AS e ON e.id = dc.entity_id
ORDER BY e.entity, dc.year
LIMIT 10;
""")

,entity,year,years_diff_fn,years_diff_builtin
0,Afghanistan,1960,66,66
1,Afghanistan,1965,61,61
2,Afghanistan,1980,46,46
3,Afghanistan,1981,45,45
4,Afghanistan,1982,44,44
5,Afghanistan,1983,43,43
6,Afghanistan,1984,42,42
7,Afghanistan,1985,41,41
8,Afghanistan,1986,40,40
9,Afghanistan,1987,39,39


### 5.2. `fn_avg_cases_per_period(cases_per_year, divider)` — середня кількість захворювань за період

- Обидва параметри й результат — `DOUBLE PRECISION`: `number_rabies` дробове, а дільник за умовою довільний (напр. `2.5`). Якби обидва аргументи були `INTEGER`, ділення було б цілочисельним (`7 / 2 = 3`).
- `NULLIF(p_divider, 0)` повертає NULL для дільника 0, тож замість помилки `division by zero` функція повертає NULL; NULL на вході так само дає NULL.
- `IMMUTABLE` — результат залежить лише від аргументів.

Перевірка на простих значеннях: `120 / 12 = 10`, ділення на 0 і NULL на вході → NULL.

In [18]:
run_sql("""
CREATE OR REPLACE FUNCTION fn_avg_cases_per_period(
    p_cases_per_year DOUBLE PRECISION,
    p_divider        DOUBLE PRECISION
)
RETURNS DOUBLE PRECISION
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT p_cases_per_year / NULLIF(p_divider, 0);
$$;

SELECT
    fn_avg_cases_per_period(120, 12) AS per_month,
    fn_avg_cases_per_period(120, 0)  AS divide_by_zero,
    fn_avg_cases_per_period(NULL, 4) AS null_input;
""")

,per_month,divide_by_zero,null_input
0,10.0,None,None


**Використання у запиті** з дільниками 12 (місяць), 4 (квартал) і 2 (півріччя). `WHERE dc.number_rabies IS NOT NULL` — лише записи з числовим значенням кількості захворювань, як вимагає умова.

In [19]:
run_sql("""
SELECT
    e.entity,
    dc.year,
    dc.number_rabies,
    fn_avg_cases_per_period(dc.number_rabies, 12) AS rabies_per_month,
    fn_avg_cases_per_period(dc.number_rabies, 4)  AS rabies_per_quarter,
    fn_avg_cases_per_period(dc.number_rabies, 2)  AS rabies_per_half_year
FROM disease_cases AS dc
JOIN entities AS e ON e.id = dc.entity_id
WHERE dc.number_rabies IS NOT NULL
ORDER BY e.entity, dc.year
LIMIT 10;
""")

,entity,year,number_rabies,rabies_per_month,rabies_per_quarter,rabies_per_half_year
0,Afghanistan,1990,2.145772,0.178814,0.536443,1.072886
1,Afghanistan,1991,2.320975,0.193415,0.580244,1.160487
2,Afghanistan,1992,2.618041,0.218170,0.654510,1.309020
3,Afghanistan,1993,2.853117,0.237760,0.713279,1.426559
4,Afghanistan,1994,2.989987,0.249166,0.747497,1.494994
5,Afghanistan,1995,3.126361,0.260530,0.781590,1.563181
6,Afghanistan,1996,3.281437,0.273453,0.820359,1.640718
7,Afghanistan,1997,3.446122,0.287177,0.861530,1.723061
8,Afghanistan,1998,3.557025,0.296419,0.889256,1.778512
9,Afghanistan,1999,3.607037,0.300586,0.901759,1.803519


### Висновки до завдання 5

- Створено дві функції: `fn_years_since(INTEGER) → INTEGER` (`STABLE`) і `fn_avg_cases_per_period(DOUBLE PRECISION, DOUBLE PRECISION) → DOUBLE PRECISION` (`IMMUTABLE`); обидві використано в запитах.
- `fn_years_since` повертає ті самі значення, що й вираз із вбудованих функцій у завданні 4 (1996 → 30 на 2026-11-07).
- `fn_avg_cases_per_period` коректно ділить річну кількість на 12 / 4 / 2 (напр. Afghanistan 1990: 2.145772 → 0.178814 на місяць, 0.536443 на квартал, 1.072886 на півріччя), а для дільника 0 чи NULL на вході повертає NULL замість помилки.